# 03 — Modelling, Cross-Validation & SHAP
3 models · 5-fold CV · Feature importance · SHAP explainability

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, warnings; warnings.filterwarnings('ignore')
from data_loader import load_data, TARGET
from features import add_interactions, build_preprocessor, ALL_FEATURES
from model import split, build_models, cross_validate_all, evaluate, compute_shap, save_model
from visualize import plot_cv_results, plot_confusion_matrix, plot_feature_importance, plot_shap_summary
df = load_data(); df = add_interactions(df)
X = df[[c for c in ALL_FEATURES if c in df.columns]]; y = df[TARGET]
X_train, X_test, y_train, y_test = split(X, y)
prep = build_preprocessor()
print('Ready. Features:', X.shape[1])

## 3.1 Cross-Validation — All 3 Models

In [ ]:
models = build_models(prep)
cv_results = cross_validate_all(models, X_train, y_train)
print(cv_results.to_string(index=False))
plot_cv_results(cv_results, save=True)

## 3.2 Best Model: Random Forest — Full Evaluation

In [ ]:
rf = models['Random Forest']
rf.fit(X_train, y_train)
results = evaluate(rf, X_test, y_test)
print(f'Test Accuracy: {results["accuracy"]:.3f}')
print(f'Test F1 macro: {results["f1_macro"]:.3f}')

## 3.3 Confusion Matrix

In [ ]:
plot_confusion_matrix(results['confusion_matrix'], save=True)

## 3.4 Feature Importance (Top 15)

In [ ]:
feat_names = list(X.columns)
plot_feature_importance(rf, feat_names, top_n=15, save=True)

## 3.5 SHAP Explainability
SHAP (SHapley Additive exPlanations) — shows *why* the model predicts each firm's class.

In [ ]:
shap_vals, X_disp, explainer = compute_shap(rf, X_test, feat_names)
if shap_vals is not None:
    plot_shap_summary(shap_vals, X_disp, class_idx=2, class_name='High Productivity', save=True)
    print('\nSHAP for LOW productivity class:')
    plot_shap_summary(shap_vals, X_disp, class_idx=0, class_name='Low Productivity', save=False)

## 3.6 Gradient Boosting for comparison

In [ ]:
gb = models['Gradient Boosting']
gb.fit(X_train, y_train)
gb_results = evaluate(gb, X_test, y_test)
print(f'GB Test Accuracy: {gb_results["accuracy"]:.3f} | F1: {gb_results["f1_macro"]:.3f}')

## 3.7 Save Best Model

In [ ]:
best = rf if results['f1_macro'] >= gb_results['f1_macro'] else gb
best_name = 'rf' if results['f1_macro'] >= gb_results['f1_macro'] else 'gb'
save_model(best, f'wbes_germany_v3_{best_name}')
print(f'Best model: {best_name}')